# Gold Layer - Fact: Sales

This notebook builds the Gold sales fact table from Silver sales and related dimensions.

## Tools and Operations
- **Spark SQL + PySpark** for fact-table shaping
- Null checks for core fact attributes
- Write Gold fact table for analytics consumption

## Output
- `workspace.gold.sales`


## 1) Inspect Input Tables


In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10

## 2) Transformations


In [0]:
%python
import pyspark.sql.functions as F

In [0]:
spark.sql("USE CATALOG workspace")

query = """
SELECT
  s.order_id,
  COALESCE(s.product_key, "Unknown") AS product_key,
  COALESCE(c.customer_key, "Unknown") AS customer_key,
  s.price,
  s.quantity,
  s.sales_amount,
  s.order_date,
  s.ship_date,
  s.due_date
FROM silver.crm_sales s
LEFT JOIN gold.customers c ON s.customer_id = c.customer_id
"""

df = spark.sql(query)

print(df.count())
df.limit(10).display()

### Null Checks


In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))).alias(c)
    for c in df.columns
]).display()

## 3) Write Gold Table


In [0]:
df.write.mode("overwrite").saveAsTable("workspace.gold.sales")

## 4) Validation


In [0]:
%sql
SELECT * FROM workspace.gold.sales LIMIT 10